# Smoot–Hawley Tariff Act & the Great Depression: Quantitative Analysis

## 1. Core Research Question & Hypotheses

**Primary question:** Did the Smoot–Hawley Tariff Act amplify the Great Depression?

**Testable hypotheses:**
- **H1:** Trade volume declined significantly after June 1930.
- **H2:** Industrial production decline correlates with trade contraction.
- **H3:** Stock market volatility increased post-enactment.

**Transmission channel:** Tariff ↑ → Imports ↓ → Retaliation ↑ → Exports ↓ → Output ↓ → Unemployment ↑

## 2. Historical & Theoretical Foundation (Brief)

- **Post-WWI:** Agricultural overproduction; protectionist political climate.
- **Policy:** President Herbert Hoover signed Smoot–Hawley (June 1930); tariff increases on thousands of goods.
- **International retaliation:** Trading partners raised tariffs; collapse in global trade.
- **Channel (analytical):** Tariff ↑ → Imports ↓ → Retaliation ↑ → Exports ↓ → Output ↓ → Unemployment ↑.

We test whether 1930 represents a **structural break** and whether **trade** is associated with **output** and **market volatility**.

## 3. Data Collection & Preparation

In [ ]:
import sys
from pathlib import Path

# Colab: mount Drive and set project root. Local: use cwd.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Quant')  # change if your folder is elsewhere
    !pip install -q pandas numpy scipy matplotlib seaborn statsmodels
except Exception:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

from src.data_prep import prepare_analysis_dataset, generate_synthetic_data, DATA_DIR

In [ ]:
# Ensure data exist: use synthetic if CSVs missing (for demo)
if not (DATA_DIR / "macro_1925_1935.csv").exists():
    macro, market = generate_synthetic_data()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    macro.to_csv(DATA_DIR / "macro_1925_1935.csv", index=False)
    market.to_csv(DATA_DIR / "dow_1925_1935.csv", index=False)

df = prepare_analysis_dataset(data_dir=DATA_DIR)
df.index = pd.to_datetime(df.index)
df = df.sort_index()
print("Post_1930 dummy: 1 after June 1930.")
df.head(10)

In [ ]:
df.describe()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

if 'imports' in df.columns and 'exports' in df.columns:
    axes[0,0].plot(df.index, df['imports'], label='Imports')
    axes[0,0].plot(df.index, df['exports'], label='Exports')
    axes[0,0].axvline(pd.Timestamp('1930-06-01'), color='red', linestyle='--', alpha=0.7, label='Tariff (Jun 1930)')
    axes[0,0].set_title('Trade (1925–1935)')
    axes[0,0].legend()
    axes[0,0].set_xlabel('')

if 'ind_prod' in df.columns:
    axes[0,1].plot(df.index, df['ind_prod'], color='green')
    axes[0,1].axvline(pd.Timestamp('1930-06-01'), color='red', linestyle='--', alpha=0.7)
    axes[0,1].set_title('Industrial Production')
    axes[0,1].set_xlabel('')

if 'dow_jones' in df.columns:
    axes[1,0].plot(df.index, df['dow_jones'], color='blue')
    axes[1,0].axvline(pd.Timestamp('1930-06-01'), color='red', linestyle='--', alpha=0.7)
    axes[1,0].set_title('Dow Jones Index')
    axes[1,0].set_xlabel('')

if 'dow_volatility' in df.columns:
    axes[1,1].plot(df.index, df['dow_volatility'], color='purple')
    axes[1,1].axvline(pd.Timestamp('1930-06-01'), color='red', linestyle='--', alpha=0.7)
    axes[1,1].set_title('Market Volatility (12m rolling std)')
    axes[1,1].set_xlabel('')

plt.tight_layout()
plt.suptitle('Pre vs Post 1930: Trade, Industrial Production, Market', y=1.02)
plt.show()

In [ ]:
pre = df[df['Post_1930'] == 0]
post = df[df['Post_1930'] == 1]

trade_cols = [c for c in ['imports', 'exports'] if c in df.columns]
print('Mean before vs after 1930')
for c in trade_cols + ['ind_prod', 'gdp', 'dow_volatility']:
    if c not in df.columns: continue
    m_pre, m_post = pre[c].mean(), post[c].mean()
    pct = (m_post - m_pre) / m_pre * 100 if m_pre != 0 else 0
    print(f"  {c}: pre={m_pre:.2f}, post={m_post:.2f}, % chg={pct:.1f}%")

print('\nVariance (post/pre ratio)')
for c in trade_cols + ['ind_prod', 'dow_jones_log_return']:
    if c not in df.columns: continue
    v_pre, v_post = pre[c].var(), post[c].var()
    if v_pre > 0: print(f"  {c}: {v_post/v_pre:.2f}")

## 5. Econometric & Statistical Analysis

### A) Structural Break Testing

**Chow test at 1930:** Split sample at June 1930; test if coefficients differ significantly.

**Variance shift:** Compare residual variance pre vs post.

In [ ]:
def chow_test(y, X, break_idx):
    """Chow test for structural break at break_idx."""
    n = len(y)
    if break_idx <= 2 or break_idx >= n - 2:
        return None
    X1, y1 = X.iloc[:break_idx], y.iloc[:break_idx]
    X2, y2 = X.iloc[break_idx:], y.iloc[break_idx:]
    X1 = sm.add_constant(X1)
    X2 = sm.add_constant(X2)
    X_all = sm.add_constant(X)
    m1 = sm.OLS(y1, X1).fit()
    m2 = sm.OLS(y2, X2).fit()
    m_pooled = sm.OLS(y, X_all).fit()
    RSS_p = m1.ssr + m2.ssr
    RSS_u = m_pooled.ssr
    k = X.shape[1] + 1
    F = (RSS_u - RSS_p) / RSS_p * (n - 2*k) / k
    from scipy.stats import f as f_dist
    pval = 1 - f_dist.cdf(F, k, n - 2*k)
    return {'F': F, 'pvalue': pval, 'RSS_pooled': RSS_p, 'RSS_unrestricted': RSS_u}

work = df.dropna(subset=['gdp', 'imports', 'exports', 'Post_1930'])
if len(work) > 10:
    y = work['gdp']
    X = work[['imports', 'exports', 'Post_1930']]
    break_idx = max(1, (work.index < '1930-06-01').sum())
    chow = chow_test(y, X, break_idx)
    if chow:
        print('Chow test (break at Jun 1930):')
        print(f"  F = {chow['F']:.3f}, p-value = {chow['pvalue']:.4f}")
        print('  Reject H0 (no break) at 5%:', chow['pvalue'] < 0.05)
else:
    print('Insufficient data for Chow test.')

In [ ]:
# Rolling regression: coefficient on trade over time
roll_window = 24
if 'gdp' in df.columns and 'imports' in df.columns and len(df) >= roll_window:
    work = df[['gdp', 'imports', 'exports']].dropna()
    if len(work) >= roll_window:
        betas = []
        dates = []
        for i in range(roll_window, len(work)):
            sub = work.iloc[i-roll_window:i]
            y = sub['gdp']
            X = sm.add_constant(sub[['imports', 'exports']])
            res = sm.OLS(y, X).fit()
            betas.append(res.params['imports'])
            dates.append(work.index[i])
        roll_df = pd.DataFrame({'date': dates, 'beta_imports': betas})
        roll_df.set_index('date').plot(title='Rolling coefficient: GDP on Imports (24m window)')
        plt.axvline(pd.Timestamp('1930-06-01'), color='red', linestyle='--', alpha=0.7)
        plt.ylabel('Coefficient')
        plt.show()

### B) Regression Analysis

- **Basic:** GDP_t = β₀ + β₁ Trade_t + β₂ Post1930 + ε_t
- **Extended:** IndustrialProduction_t = β₀ + β₁ Exports_t + β₂ Imports_t + β₃ Post1930 + ε_t

In [ ]:
work = df.dropna(subset=['gdp', 'imports', 'exports', 'Post_1930'])
if len(work) > 5:
    y = work['gdp']
    X = sm.add_constant(work[['imports', 'exports', 'Post_1930']])
    m1 = sm.OLS(y, X).fit()
    print('Model: GDP_t = β0 + β1*Imports + β2*Exports + β3*Post_1930')
    print(m1.summary())

In [ ]:
work = df.dropna(subset=['ind_prod', 'imports', 'exports', 'Post_1930'])
if len(work) > 5:
    y = work['ind_prod']
    X = sm.add_constant(work[['imports', 'exports', 'Post_1930']])
    m2 = sm.OLS(y, X).fit()
    print('Model: IndustrialProduction_t = β0 + β1*Imports + β2*Exports + β3*Post_1930')
    print(m2.summary())

### C) Event Study (Market Angle)

Dow Jones returns and volatility around June 1930.

In [ ]:
if 'dow_jones_log_return' in df.columns and 'dow_volatility' in df.columns:
    event = pd.Timestamp('1930-06-01')
    window = 6  # months before/after
    mask = (df.index >= event - pd.DateOffset(months=window)) & (df.index <= event + pd.DateOffset(months=window))
    ev = df.loc[mask, ['dow_jones_log_return', 'dow_volatility']].copy()
    ev['abnormal_return'] = ev['dow_jones_log_return'] - ev['dow_jones_log_return'].mean()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].bar(range(len(ev)), ev['dow_jones_log_return']*100)
    ax[0].axvline(len(ev)//2, color='red', linestyle='--', label='Jun 1930')
    ax[0].set_title('Dow Jones log returns around event')
    ax[0].set_ylabel('%')
    ax[1].plot(ev.index, ev['dow_volatility'])
    ax[1].axvline(event, color='red', linestyle='--')
    ax[1].set_title('Volatility around Jun 1930')
    plt.tight_layout()
    plt.show()

## 6. Causality & Robustness (Optional)

- **Granger causality:** Does trade (lagged) help predict GDP?
- **Lagged trade effects:** Include lags of exports/imports in the regression.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

gc_data = df[['gdp', 'imports']].dropna()
if len(gc_data) > 24:
    try:
        # Test: do lags of 'imports' Granger-cause 'gdp'?
        out = grangercausalitytests(gc_data[['gdp', 'imports']], maxlag=3, verbose=False)
        for lag, (ssr_ftest, ssr_chi2, lrt, params_ftest) in out.items():
            print(f'Lag {lag}: F_pval={params_ftest["ssr_ftest"][1]:.4f}')
    except Exception as e:
        print('Granger test:', e)
else:
    print('Insufficient observations for Granger.')

## 8. Advanced Econometrics (Advanced Version)

- **CUSUM:** Parameter stability (Brown–Durbin–Evans); reject H0 ⇒ coefficients unstable.
- **Newey–West HAC:** Robust standard errors for autocorrelation/heteroskedasticity.
- **VAR + Granger + IRF:** Multivariate causality and impulse response functions.
- **Placebo event study:** Run event study at a non-event date (e.g. June 1928); should show no effect.
- **Difference-in-differences:** US (treatment) vs synthetic control (no 1930 break); DiD = interaction effect.

### 8.1 CUSUM stability test

Tests whether regression coefficients are stable over time. Rejection ⇒ structural instability (consistent with a break).

In [ ]:
from src.advanced_analysis import cusum_test

work = df.dropna(subset=['gdp', 'imports', 'exports', 'Post_1930'])
if len(work) > 20:
    y, X = work['gdp'], work[['imports', 'exports', 'Post_1930']]
    cusum = cusum_test(y, X, alpha=0.05)
    if 'error' not in cusum:
        print('CUSUM test (GDP ~ imports + exports + Post_1930):')
        print('  Reject stability (structural instability):', cusum['rejects_stability'])
        n_c = len(cusum['cusum'])
        plt.figure(figsize=(10, 4))
        plt.plot(range(n_c), cusum['cusum'], label='CUSUM')
        plt.plot(range(n_c), cusum['band_lower'], 'k--', alpha=0.7)
        plt.plot(range(n_c), cusum['band_upper'], 'k--', alpha=0.7)
        plt.axvline(pd.Timestamp('1930-06-01'), color='red', linestyle=':', alpha=0.8)
        plt.legend(); plt.title('CUSUM statistic and confidence band'); plt.tight_layout(); plt.show()
    else:
        print('CUSUM:', cusum.get('error', cusum))
else:
    print('Insufficient data for CUSUM.')

### 8.2 Newey–West HAC standard errors

OLS with heteroskedasticity- and autocorrelation-consistent (HAC) standard errors. Appropriate for time series.

In [ ]:
from src.advanced_analysis import ols_hac

work = df.dropna(subset=['gdp', 'imports', 'exports', 'Post_1930'])
if len(work) > 10:
    y, X = work['gdp'], work[['imports', 'exports', 'Post_1930']]
    res_hac = ols_hac(y, X, maxlags=int(len(y)**0.25))
    print('GDP ~ Imports + Exports + Post_1930 (Newey-West HAC SEs):')
    print(res_hac.summary())

### 8.3 VAR: Granger causality and impulse response

Vector autoregression on (e.g.) GDP, imports, exports; test Granger causality and plot impulse responses.

In [ ]:
from src.advanced_analysis import var_granger_irf

var_data = df[['gdp', 'imports', 'exports']].dropna()
if len(var_data) > 30:
    out = var_granger_irf(var_data, variables=['gdp', 'imports', 'exports'], maxlags=3, irf_periods=12)
    if 'error' not in out:
        print('VAR-based Granger causality (F-test p-values):')
        for k, v in out['granger'].items():
            print(f"  {k}: p={v['pvalue']:.4f}, reject={v['reject']}")
        if 'error' not in out.get('irf', {}):
            fig, ax = plt.subplots(1, 1, figsize=(8, 4))
            for var, irf_series in out['irf'].items():
                if isinstance(irf_series, np.ndarray):
                    ax.plot(irf_series, label=var)
            ax.axhline(0, color='gray', linestyle='--'); ax.legend(); ax.set_title('IRF (orthogonalized, response to shocks)'); plt.tight_layout(); plt.show()
    else:
        print('VAR:', out.get('error'))
else:
    print('Insufficient data for VAR.')

### 8.4 Placebo event study

Run the same event study around a **non-event** date (e.g. June 1928). We should not see a similar volatility spike or return pattern; if we do, the original event study may be picking up trend rather than the tariff.

In [ ]:
from src.advanced_analysis import placebo_event_study

if 'dow_jones_log_return' in df.columns:
    real_mean, real_std, real_ret = placebo_event_study(df['dow_jones_log_return'], '1930-06-01', 6)
    place_mean, place_std, place_ret = placebo_event_study(df['dow_jones_log_return'], '1928-06-01', 6)
    print('Event study comparison:')
    print('  Real event (Jun 1930): mean return = %.4f, vol = %.4f' % (real_mean, real_std))
    print('  Placebo (Jun 1928):   mean return = %.4f, vol = %.4f' % (place_mean, place_std))
    print('  If tariff mattered, real event window should show stronger vol or more negative returns than placebo.')
else:
    print('Dow returns not available.')

### 8.5 Difference-in-differences (DiD)

Compare **treatment** (US trade/output) with a **control** series that has no 1930 break (synthetic "rest of world"). DiD coefficient = effect of treatment in post period. Builds long-format data: treat=1 for US, treat=0 for control; post=1 after Jun 1930.

In [ ]:
from src.advanced_analysis import did_estimate
from src.data_prep import generate_control_series

# Long format: US (treat=1) vs synthetic control (treat=0), post=1 after Jun 1930
control_df = generate_control_series(df.index).set_index('date')
post = (df.index >= pd.Timestamp('1930-06-01')).astype(int)
long_us = pd.DataFrame({'outcome': df['imports'], 'treat': 1, 'post': post})
long_control = pd.DataFrame({'outcome': control_df['control_imports'], 'treat': 0, 'post': post})
combined = pd.concat([long_us.dropna(subset=['outcome']), long_control.dropna(subset=['outcome'])], axis=0)
if len(combined) > 20:
    res_did = did_estimate(combined, 'outcome', 'treat', 'post')
    print('DiD: outcome (imports) ~ treat + post + treat×post')
    print('  DiD coefficient (treat×post) = effect of treatment in post period:', res_did.params.get('_did', np.nan))
    print(res_did.summary().tables[1])
else:
    print('Insufficient data for DiD.')

## 7. Discussion & Interpretation

- **Did tariffs cause the trade collapse or amplify an existing one?** (Interpret Chow + regressions.)
- **Was policy reactive or destructive?** (Context: Depression had started before June 1930.)
- **Comparison to modern trade wars:** Regime shifts and volatility.

*(Fill in with your results and narrative.)*